In [0]:
# !pip install databricks-feature-engineering
# dbutils.library.restartPython()

In [0]:

import pyspark

catalogo_esquema = "workspace.gold"
tabela_gold = f"{catalogo_esquema}.base_eventos_produto_gold_particionada"

# Execução do pipeline via Spark SQL
spark.sql(f"""
CREATE OR REPLACE TABLE {tabela_gold}
    USING DELTA
    PARTITIONED BY (dia_prtc)
    AS
        -- Cruzamento da Silver com a Dimensão de Produtos
        SELECT 
            s.id_usuario,
            s.tipo_evento,
            s.dispositivo,
            s.sistema_operacional,
            s.timestamp_registro,
            s.origem_trafego,
            s.idade,
            s.genero,
            s.renda_mensal_estimada_k,
            s.regiao,
            s.tempo_clique_segundos,
            s.interacoes_chat_suporte,
            s.cupons_ativos_conta,
            s.score_satisfacao_nps,
            s.dias_desde_ultima_visita,
            p.categoria,
            p.preco_reais,
        -- Extrai o carimbo de data/hora no formato yyyymmdd e converte para INT
        CAST(date_format(s.timestamp_registro, 'yyyyMMdd') AS INT) AS dia_prtc
        FROM workspace.silver.tabela_eventos_silver s
        INNER JOIN  workspace.default.base_dim_produtos p ON s.id_produto = p.id_produto
""")

print("Camada Gold consolidada com sucesso!")
display(spark.sql(f"SELECT * FROM {tabela_gold}"))

# prompt para o Genie
Com base na tabela da camada gold, base_eventos_produto_gold_particionada que contém as colunas id_usuario, tipo_evento, dispositivo, sistema_operacional,
 origem_trafego, idade, genero, renda_mensal_estimada_k,regiao,
 tempo_clique_segundos, interacoes_chat_suporte, cupons_ativos_conta,score_satisfacao_nps, dias_desde_ultima_visita, categoria, preco_reais. A tabela é particinada diaramente e tem a coluna timestamp_registro também. Faça apenas sugestão de novas features que podem ser utes para a criação de um modelo de machine learning que irá prever a compra da categoria eletronicos. Não há necessidade de criar código, apenas fale de features que pode ser interessantes criar.

In [0]:
from databricks import feature_engineering
from pyspark.sql.functions import col, sum, avg, max, first, lower, trim

print(f"Carregando a tabela Gold atualizada...")
df_gold = spark.table(tabela_gold)

print("Agregando features mantendo a granularidade temporal do evento...")

# Agrega os dados incluindo o timestamp para controle histórico da Feature Store
df_user_features_temporal = df_gold.groupBy("id_usuario", "timestamp_registro").agg(
    # Métricas demográficas higienizadas
    lower(trim(first("idade"))).alias("idade_cliente"),
    lower(trim(first("genero"))).alias("genero_cliente"),
    lower(trim(first("regiao"))).alias("regiao_cliente"),
    first("renda_mensal_estimada_k").alias("renda_mensal_k"),
    
    # Métricas comportamentais e de engajamento
    avg("tempo_clique_segundos").alias("tempo_medio_clique_segundos"),
    max("interacoes_chat_suporte").alias("total_interacoes_suporte"),
    max("cupons_ativos_conta").alias("cupons_ativos"),
    max("score_satisfacao_nps").alias("score_nps_cliente"),
    max("dias_desde_ultima_visita").alias("dias_inatividade"),
    
    # Valor financeiro acumulado
    sum(col("preco_reais")).alias("total_gasto_acumulado_reais")
)
df_user_features_temporal.display()

In [0]:

print(f"Carregando a tabela Gold...")
df_gold = spark.table(tabela_gold)
print("Agregando features mantendo a granularidade temporal do evento")

# Agrega os dados mantendo a coluna de timestamp
df_user_features_temporal = df_gold.groupBy("id_usuario", "timestamp_registro").agg(
    lower(trim(first("idade"))).alias("idade_cliente"),
    lower(trim(first("genero"))).alias("genero_cliente"),
    lower(trim(first("regiao"))).alias("regiao_cliente"),
    first("renda_mensal_estimada_k").alias("renda_mensal_k"),
    avg("tempo_clique_segundos").alias("tempo_medio_clique_segundos"),
    avg("interacoes_chat_suporte").alias("media_interacoes_suporte"),
    avg("cupons_ativos_conta").alias("media_cupons_ativos"),
    avg("score_satisfacao_nps").alias("media_score_nps_cliente"),
    avg("dias_desde_ultima_visita").alias("media_dias_inatividade"),
    sum(col("preco_reais")).alias("total_gasto_acumulado_reais")
)

# Feature Store
fe = feature_engineering.FeatureEngineeringClient()
nome_feature_table_temporal = f"{catalogo_esquema}.user_profile_features_temporal"

# Podemos criar como chave composta, pelo id_usuario e o timestamp
fe.create_table(
    name=nome_feature_table_temporal,
    primary_keys=["id_usuario","timestamp_registro"], 
    timeseries_columns=["timestamp_registro"],
    df=df_user_features_temporal,
    schema=df_user_features_temporal.schema,
    description="Perfil de recursos comportamentais e demográficos históricos por usuário, otimizada para point-in-time correctness."
)
